# Energy-Aware GPU Recommender — Data Cleaning & EDA

**Goal:** Produce clean, validated, analysis-ready datasets from:
1. Kaggle PC Video Game Requirements v2 (`videogame_requirements.csv`)
2. TechPowerUp GPU Specs (`gpu_specs.csv`)

_No ML models are trained here. The sole focus is data preparation and EDA._

---

## Contents
1. [Setup](#1-setup)
2. [Load raw data](#2-load-raw-data)
3. [Clean game requirements](#3-clean-game-requirements)
4. [Clean GPU specs](#4-clean-gpu-specs)
5. [Validate cleaned data](#5-validate-cleaned-data)
6. [Save outputs](#6-save-outputs)
7. [Exploratory Data Analysis](#7-exploratory-data-analysis)

## 1 Setup

In [ ]:
import sys, os

# Make sure the project root is on the path when running from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

from src.data_cleaning import (
    clean_game_requirements,
    clean_gpu_specs,
    NUMERIC_GPU_COLS,
    GPU_SPECS_COLS,
)
from src.validation import (
    run_all_validations,
    missing_value_summary,
)

# Paths
RAW_DIR        = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR  = os.path.join(PROJECT_ROOT, "data", "processed")
FIGURES_DIR    = os.path.join(PROJECT_ROOT, "reports", "figures")

GAME_REQ_PATH  = os.path.join(RAW_DIR, "videogame_requirements.csv")
GPU_SPECS_PATH = os.path.join(RAW_DIR, "gpu_specs.csv")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR,   exist_ok=True)

print("Setup complete.")

## 2 Load raw data

> Place the following files in `data/raw/` before running this section:
> - `videogame_requirements.csv` (Kaggle PC Video Game Requirements v2)
> - `gpu_specs.csv` (TechPowerUp GPU specs — scraped or manually exported)

In [ ]:
# Load raw datasets for initial inspection
raw_games = pd.read_csv(GAME_REQ_PATH)
raw_gpu   = pd.read_csv(GPU_SPECS_PATH)

print("=== Game requirements raw shape:", raw_games.shape)
raw_games.head(3)

In [ ]:
print("=== GPU specs raw shape:", raw_gpu.shape)
raw_gpu.head(3)

In [ ]:
print("=== Game requirements dtypes")
print(raw_games.dtypes)

print("\n=== GPU specs dtypes")
print(raw_gpu.dtypes)

In [ ]:
print("=== Missing values — raw game requirements")
print(raw_games.isna().sum().sort_values(ascending=False))

print("\n=== Missing values — raw GPU specs")
print(raw_gpu.isna().sum().sort_values(ascending=False))

## 3 Clean game requirements

In [ ]:
games_min, games_rec = clean_game_requirements(GAME_REQ_PATH)

print("\n--- Minimum requirements (sample) ---")
display(games_min.head())

print("\n--- Recommended requirements (sample) ---")
display(games_rec.head())

In [ ]:
print("games_min shape:", games_min.shape)
print("games_rec shape:", games_rec.shape)
print("\ngames_min dtypes\n", games_min.dtypes)
print("\ngames_rec dtypes\n", games_rec.dtypes)

## 4 Clean GPU specs

In [ ]:
gpu_specs = clean_gpu_specs(GPU_SPECS_PATH)

print("\n--- GPU specs (sample) ---")
display(gpu_specs.head())
print("\nShape:", gpu_specs.shape)
print("\nDtypes\n", gpu_specs.dtypes)

In [ ]:
print("GPU specs describe:")
display(gpu_specs.describe())

## 5 Validate cleaned data

In [ ]:
GPU_REQUIRED_COLS  = ["gpu_name", "psu_watts"]
GAME_REQUIRED_COLS = ["game_title"]

validation_results = run_all_validations(
    gpu_specs=gpu_specs,
    games_min=games_min,
    games_rec=games_rec,
    gpu_required_cols=GPU_REQUIRED_COLS,
    game_required_cols=GAME_REQUIRED_COLS,
    numeric_gpu_cols=NUMERIC_GPU_COLS,
)

In [ ]:
print("\n=== Missing value summary — GPU specs ===")
display(missing_value_summary(gpu_specs))

print("\n=== Missing value summary — games_min ===")
display(missing_value_summary(games_min))

print("\n=== Missing value summary — games_rec ===")
display(missing_value_summary(games_rec))

## 6 Save outputs

In [ ]:
games_min.to_csv(os.path.join(PROCESSED_DIR, "games_min_clean.csv"),  index=False)
games_rec.to_csv(os.path.join(PROCESSED_DIR, "games_rec_clean.csv"),  index=False)
gpu_specs.to_csv(os.path.join(PROCESSED_DIR, "gpu_specs_clean.csv"),  index=False)

print("Saved cleaned files to", PROCESSED_DIR)

## 7 Exploratory Data Analysis

### 7.1 Missing value summary (heatmap)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, df, title in zip(
    axes,
    [gpu_specs, games_min, games_rec],
    ["GPU Specs", "Games — Min Requirements", "Games — Rec Requirements"],
):
    null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    null_pct.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Missing values (%) — {title}")
    ax.set_xlabel("% null")
    ax.axvline(x=0, color="grey", linewidth=0.8)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "missing_value_summary.png"), bbox_inches="tight")
plt.show()

### 7.2 Distribution of PSU requirements

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, df, title in zip(
    axes,
    [gpu_specs, games_min, games_rec],
    ["GPU Specs PSU", "Games Min PSU", "Games Rec PSU"],
):
    if "psu_watts" not in df.columns:
        ax.set_visible(False)
        continue
    psu = df["psu_watts"].dropna()
    ax.hist(psu, bins=30, color="coral", edgecolor="white", linewidth=0.6)
    ax.set_title(f"PSU Distribution — {title}")
    ax.set_xlabel("Watts")
    ax.set_ylabel("Count")
    ax.axvline(psu.median(), color="navy", linestyle="--", label=f"Median {psu.median():.0f}W")
    ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "psu_distributions.png"), bbox_inches="tight")
plt.show()

### 7.3 Minimum vs recommended PSU comparison

In [ ]:
if "psu_watts" in games_min.columns and "psu_watts" in games_rec.columns:
    fig, ax = plt.subplots(figsize=(9, 5))

    psu_min = games_min["psu_watts"].dropna()
    psu_rec = games_rec["psu_watts"].dropna()

    ax.hist(psu_min, bins=30, alpha=0.6, label="Minimum",     color="steelblue",  edgecolor="white")
    ax.hist(psu_rec, bins=30, alpha=0.6, label="Recommended", color="darkorange", edgecolor="white")
    ax.set_title("Min vs Recommended PSU Requirement")
    ax.set_xlabel("Watts")
    ax.set_ylabel("Count")
    ax.legend()

    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "min_vs_rec_psu.png"), bbox_inches="tight")
    plt.show()
else:
    print("PSU column not available for comparison.")

### 7.4 Distribution of GPU memory size

In [ ]:
if "memory_size_mb" in gpu_specs.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    mem = gpu_specs["memory_size_mb"].dropna() / 1024  # convert to GB for readability
    ax.hist(mem, bins=25, color="mediumseagreen", edgecolor="white", linewidth=0.6)
    ax.set_title("GPU VRAM Distribution")
    ax.set_xlabel("Memory Size (GB)")
    ax.set_ylabel("Count")
    ax.axvline(mem.median(), color="darkred", linestyle="--", label=f"Median {mem.median():.1f} GB")
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "gpu_memory_size_dist.png"), bbox_inches="tight")
    plt.show()

### 7.5 Distribution / top memory types

In [ ]:
if "memory_type" in gpu_specs.columns:
    mem_type_counts = gpu_specs["memory_type"].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(9, 5))
    mem_type_counts.plot(kind="barh", ax=ax, color="mediumpurple", edgecolor="white")
    ax.invert_yaxis()
    ax.set_title("Top GPU Memory Types")
    ax.set_xlabel("Count")
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "memory_type_dist.png"), bbox_inches="tight")
    plt.show()

    print("\nMemory type counts:")
    print(mem_type_counts.to_string())

### 7.6 GPU feature ranges

In [ ]:
numeric_cols_present = [c for c in NUMERIC_GPU_COLS if c in gpu_specs.columns]

print("=== GPU feature summary statistics ===")
display(gpu_specs[numeric_cols_present].describe().T)

In [ ]:
# Box plots for GPU numeric features
n_cols = len(numeric_cols_present)
if n_cols > 0:
    fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]

    for ax, col in zip(axes, numeric_cols_present):
        data = gpu_specs[col].dropna()
        ax.boxplot(data, vert=True, patch_artist=True,
                   boxprops=dict(facecolor="lightblue", color="navy"),
                   medianprops=dict(color="darkred", linewidth=2))
        ax.set_title(col.replace("_", " ").title(), fontsize=9)
        ax.set_xticks([])

    plt.suptitle("GPU Numeric Feature Ranges", fontsize=12, y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "gpu_feature_ranges.png"), bbox_inches="tight")
    plt.show()

### 7.7 Correlation heatmap of numeric GPU features

In [ ]:
if len(numeric_cols_present) >= 2:
    corr = gpu_specs[numeric_cols_present].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        square=True,
        linewidths=0.5,
        ax=ax,
        vmin=-1,
        vmax=1,
    )
    ax.set_title("Correlation Heatmap — GPU Numeric Features")
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "gpu_correlation_heatmap.png"), bbox_inches="tight")
    plt.show()
else:
    print("Not enough numeric columns for a correlation heatmap.")

---

## Summary

| Dataset | Rows | Columns |
|---------|------|---------|
| `gpu_specs_clean.csv` | `{gpu}` | `{gpu_c}` |
| `games_min_clean.csv` | `{gmin}` | `{gmin_c}` |
| `games_rec_clean.csv` | `{grec}` | `{grec_c}` |

_Next step: feature engineering and ML modelling._

In [ ]:
print(f"gpu_specs_clean  : {gpu_specs.shape[0]:>5} rows × {gpu_specs.shape[1]} cols")
print(f"games_min_clean  : {games_min.shape[0]:>5} rows × {games_min.shape[1]} cols")
print(f"games_rec_clean  : {games_rec.shape[0]:>5} rows × {games_rec.shape[1]} cols")